In [20]:
import os
import torch
import pandas as pd
import scanpy as sc
import numpy as np
import glob
import shutil
import SpatialGlue
from SpatialGlue.preprocess import clr_normalize_each_cell, pca

In [ ]:
sample='sample_tmp'
binsize=100
input_path=f'/data/work/cite_seq/{sample}'

output_path=f'/data/work/cite_seq/result/{sample}/bin{binsize}'
if not os.path.exists(output_path):
    os.makedirs(output_path)

In [ ]:
RNA_path=f'{input_path}/{sample}.bin{binsize}_1.0.h5ad'
protein_path=f'{input_path}/{sample}.protein.bin{binsize}_0.1.h5ad'

In [ ]:
print(RNA_path)
print(protein_path)
raw_adata_RNA=sc.read_h5ad(RNA_path)
raw_adata_protein=sc.read_h5ad(protein_path)

In [14]:
if raw_adata_RNA.raw is not None:
    raw_adata_RNA.X = raw_adata_RNA.raw.X
    print("raw_adata_RNA.X 已恢复为原始数据。")
else:
    print("raw_adata_RNA.raw 不存在，无法恢复原始数据。")
    
if raw_adata_protein.raw is not None:
    raw_adata_protein.X = raw_adata_protein.raw.X
    print("raw_adata_protein.X 已恢复为原始数据。")
else:
    print("raw_adata_protein.raw 不存在，无法恢复原始数据。")

raw_adata_RNA.raw 不存在，无法恢复原始数据。
raw_adata_protein.raw 不存在，无法恢复原始数据。


In [15]:
n_rows, n_cols = raw_adata_RNA.X.shape
if n_rows < 100 or n_cols < 100:
    raise ValueError("The matrix is too small to extract a 100x100 submatrix.")
start_row = np.random.randint(0, n_rows - 100)
start_col = np.random.randint(0, n_cols - 100)
submatrix = raw_adata_RNA.X[start_row:start_row + 100, start_col:start_col + 100].toarray()
is_all_integers = np.all(np.equal(np.mod(submatrix, 1), 0))
print("The extracted 100x100 submatrix is all integers:", is_all_integers)

The extracted 100x100 submatrix is all integers: True


In [16]:
adata_RNA = raw_adata_RNA.copy()
adata_protein = raw_adata_protein.copy()

adata_RNA.raw=adata_RNA
adata_protein.raw=adata_protein

adata_RNA.var_names_make_unique()
adata_protein.var_names_make_unique()

In [17]:
#RNA
sc.pp.filter_genes(adata_RNA, min_cells=10)
sc.pp.filter_cells(adata_RNA, min_genes=80)
sc.pp.calculate_qc_metrics(adata_RNA, inplace=True)

sc.pp.highly_variable_genes(adata_RNA, flavor="seurat_v3", n_top_genes=3000)
adata_RNA.layers['counts'] = adata_RNA.X.copy()
sc.pp.normalize_total(adata_RNA, target_sum=1e4)
sc.pp.log1p(adata_RNA)
adata_RNA.layers['normlog1p'] = adata_RNA.X.copy()
adata_RNA_hvg = adata_RNA[:, adata_RNA.var['highly_variable']]
adata_RNA.obsm['feat'] = pca(adata_RNA_hvg, n_comps=adata_protein.n_vars-1)

#protein
sc.pp.filter_genes(adata_protein, min_cells=50)
adata_protein = adata_protein[adata_RNA.obs_names].copy()
adata_protein = clr_normalize_each_cell(adata_protein)
adata_protein.obsm['feat'] = pca(adata_protein, n_comps=adata_protein.n_vars-1)

In [18]:
adata_RNA.write(f"{output_path}/{sample}.bin100.RNA.h5ad")

adata_protein.write(f"{output_path}/{sample}.bin100.protein.h5ad")

In [19]:
print('ggg')

ggg
